# Project 3: SQL Data Analysis 🗄️
**DecodeLabs | Industrial Training Kit | Batch 2026**

**Goal:** Use SQL queries to extract insights from the e-commerce dataset.

**Tool:** Python + SQLite (built-in, no installation needed)


## Step 1: Import Libraries & Setup Database

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
print('Libraries imported successfully ✅')

## Step 2: Load Data into SQL Database

In [ ]:
# Load CSV into pandas
df = pd.read_csv('ecommerce_sql.csv', parse_dates=['Date'])

# Create SQLite database in memory
conn = sqlite3.connect('ecommerce.db')

# Load dataframe into SQL table
df.to_sql('orders', conn, if_exists='replace', index=False)

print(f'✅ Database created with {len(df):,} rows')
print(f'Table: orders')
print(f'Columns: {df.columns.tolist()}')

## Step 3: Basic SELECT Query

In [ ]:
query = """
SELECT OrderID, Date, Product, Quantity, UnitPrice, TotalPrice, OrderStatus
FROM orders
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
print('=== First 10 Orders ===')
result

## Step 4: WHERE Clause - Filter Data

In [ ]:
# Query 1: High-value orders above $1000
query = """
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
WHERE TotalPrice > 1000
ORDER BY TotalPrice DESC
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
print(f'=== High-Value Orders (> $1000) ===')
print(f'Total found: {len(pd.read_sql_query("SELECT * FROM orders WHERE TotalPrice > 1000", conn))}')
result

In [ ]:
# Query 2: Cancelled orders
query = """
SELECT OrderID, Product, TotalPrice, PaymentMethod
FROM orders
WHERE OrderStatus = 'Cancelled'
ORDER BY TotalPrice DESC
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
print('=== Cancelled Orders ===')
result

In [ ]:
# Query 3: Laptop orders with Credit Card
query = """
SELECT OrderID, CustomerID, Quantity, TotalPrice
FROM orders
WHERE Product = 'Laptop' AND PaymentMethod = 'Credit Card'
ORDER BY TotalPrice DESC
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
print('=== Laptop Orders Paid by Credit Card ===')
result

## Step 5: GROUP BY + Aggregations (COUNT, SUM, AVG)

In [ ]:
# Query 4: Revenue by Product
query = """
SELECT 
    Product,
    COUNT(*) AS Total_Orders,
    SUM(TotalPrice) AS Total_Revenue,
    AVG(TotalPrice) AS Avg_Order_Value,
    SUM(Quantity) AS Total_Units_Sold
FROM orders
GROUP BY Product
ORDER BY Total_Revenue DESC
"""

result = pd.read_sql_query(query, conn)
result['Total_Revenue'] = result['Total_Revenue'].round(2)
result['Avg_Order_Value'] = result['Avg_Order_Value'].round(2)
print('=== Revenue by Product ===')
result

In [ ]:
# Visualize Revenue by Product
plt.figure(figsize=(10, 5))
plt.bar(result['Product'], result['Total_Revenue'], color='steelblue')
plt.title('Total Revenue by Product')
plt.xlabel('Product')
plt.ylabel('Total Revenue ($)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('sql_revenue_by_product.png', dpi=150)
plt.show()
print('✅ Chart saved')

In [ ]:
# Query 5: Orders by Status
query = """
SELECT 
    OrderStatus,
    COUNT(*) AS Total_Orders,
    SUM(TotalPrice) AS Total_Revenue,
    ROUND(AVG(TotalPrice), 2) AS Avg_Value
FROM orders
GROUP BY OrderStatus
ORDER BY Total_Orders DESC
"""

result = pd.read_sql_query(query, conn)
print('=== Orders by Status ===')
result

In [ ]:
# Query 6: Revenue by Payment Method
query = """
SELECT 
    PaymentMethod,
    COUNT(*) AS Total_Orders,
    ROUND(SUM(TotalPrice), 2) AS Total_Revenue
FROM orders
GROUP BY PaymentMethod
ORDER BY Total_Revenue DESC
"""

result = pd.read_sql_query(query, conn)
print('=== Revenue by Payment Method ===')
result

## Step 6: HAVING Clause - Filter Groups

In [ ]:
# Query 7: Products with more than 150 orders
query = """
SELECT 
    Product,
    COUNT(*) AS Total_Orders,
    ROUND(SUM(TotalPrice), 2) AS Total_Revenue
FROM orders
GROUP BY Product
HAVING Total_Orders > 150
ORDER BY Total_Orders DESC
"""

result = pd.read_sql_query(query, conn)
print('=== Products with More Than 150 Orders ===')
result

## Step 7: Advanced Analysis - Monthly Revenue Trend

In [ ]:
# Query 8: Monthly Revenue
query = """
SELECT 
    STRFTIME('%Y-%m', Date) AS Month,
    COUNT(*) AS Total_Orders,
    ROUND(SUM(TotalPrice), 2) AS Monthly_Revenue,
    ROUND(AVG(TotalPrice), 2) AS Avg_Order_Value
FROM orders
GROUP BY Month
ORDER BY Month
"""

result = pd.read_sql_query(query, conn)
print('=== Monthly Revenue ===')
print(result.to_string(index=False))

In [ ]:
# Visualize Monthly Trend
plt.figure(figsize=(14, 5))
plt.plot(result['Month'], result['Monthly_Revenue'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(range(len(result)), result['Monthly_Revenue'], alpha=0.2, color='steelblue')
plt.xticks(range(len(result)), result['Month'], rotation=45)
plt.title('Monthly Revenue Trend (SQL Query)')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.tight_layout()
plt.savefig('sql_monthly_trend.png', dpi=150)
plt.show()
print('✅ Chart saved')

## Step 8: Top Customers Analysis

In [ ]:
# Query 9: Top 10 Customers by Revenue
query = """
SELECT 
    CustomerID,
    COUNT(*) AS Total_Orders,
    ROUND(SUM(TotalPrice), 2) AS Total_Spent,
    ROUND(AVG(TotalPrice), 2) AS Avg_Order_Value
FROM orders
GROUP BY CustomerID
ORDER BY Total_Spent DESC
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
print('=== Top 10 Customers by Revenue ===')
result

## Step 9: Executive Summary

In [ ]:
# Final Summary Query
query = """
SELECT
    COUNT(*) AS Total_Orders,
    ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
    ROUND(AVG(TotalPrice), 2) AS Avg_Order_Value,
    ROUND(MAX(TotalPrice), 2) AS Max_Order,
    ROUND(MIN(TotalPrice), 2) AS Min_Order,
    COUNT(DISTINCT CustomerID) AS Unique_Customers,
    COUNT(DISTINCT Product) AS Unique_Products
FROM orders
"""

result = pd.read_sql_query(query, conn)

print('=' * 50)
print('      EXECUTIVE SUMMARY - SQL ANALYSIS')
print('=' * 50)
print(f'Total Orders:       {result["Total_Orders"][0]:,}')
print(f'Total Revenue:      ${result["Total_Revenue"][0]:,}')
print(f'Avg Order Value:    ${result["Avg_Order_Value"][0]:,}')
print(f'Max Order:          ${result["Max_Order"][0]:,}')
print(f'Min Order:          ${result["Min_Order"][0]:,}')
print(f'Unique Customers:   {result["Unique_Customers"][0]:,}')
print(f'Unique Products:    {result["Unique_Products"][0]:,}')
print('=' * 50)

conn.close()
print('\n✅ Database connection closed')
print('✅ Project 3 Complete!')